# 06 -- Privacy by Architecture

**Author**: Tamer Atesyakar

We examine the three concrete privacy guarantees of I3:
1. **No raw text at rest** -- the interaction diary stores only derived summaries.
2. **PII sanitisation** -- a regex filter scrubs emails, phone numbers, URLs, and card-like digit runs before *any* content crosses a trust boundary.
3. **At-rest encryption** -- Fernet round-trip on all database fields tagged sensitive.

**Citations**
- Sweeney (2002). *k-Anonymity: A Model for Protecting Privacy.* IJUFKS.
- Bernstein (2011). *Extending the Salsa20 nonce.* (underpins Fernet's AES-CBC + HMAC-SHA256).
- Dwork & Roth (2014). *The Algorithmic Foundations of Differential Privacy.*

In [ ]:
import re
import json
import sqlite3
import tempfile
import os
import pprint

## 6.1 The diary SQLite schema

The runtime diary stores *summaries and tags*, not the original user text. Below we construct an in-memory version of the schema and verify that the content column is always derived, never raw.

In [ ]:
SCHEMA = '''
CREATE TABLE diary (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    ts         REAL    NOT NULL,
    user_id    TEXT    NOT NULL,
    summary    BLOB    NOT NULL,    -- Fernet-encrypted derived summary
    emotion    TEXT,
    topics     TEXT,                -- JSON array of short tags
    -- NOTE: no `raw_text` column exists by design
    UNIQUE(ts, user_id)
);
'''
conn = sqlite3.connect(':memory:')
conn.executescript(SCHEMA)
cols = [r[1] for r in conn.execute('PRAGMA table_info(diary)').fetchall()]
print('columns:', cols)
assert 'raw_text' not in cols and 'message' not in cols, 'raw-text leakage!'
print('invariant holds: no raw-text columns in schema')

## 6.2 The PII sanitiser

Hand-crafted regex set covering email, phone, URL, and card-like digit sequences. In production this is `i3.privacy.sanitizer.scrub`.

In [ ]:
RULES = [
    ('EMAIL', re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}')),
    ('URL',   re.compile(r'https?://\\S+')),
    ('PHONE', re.compile(r'(?<!\\d)(\\+?\\d[\\d\\s().-]{7,}\\d)(?!\\d)')),
    ('CARD',  re.compile(r'(?<!\\d)(?:\\d[ -]?){13,19}(?!\\d)')),
    ('SSN',   re.compile(r'(?<!\\d)\\d{3}-\\d{2}-\\d{4}(?!\\d)')),
]

def scrub(text: str) -> tuple[str, dict]:
    out = text; counts = {}
    for tag, pat in RULES:
        out, n = pat.subn(f'<{tag}>', out)
        if n: counts[tag] = n
    return out, counts

samples = [
    'Contact me at tamer@example.com or 4242 4242 4242 4242',
    'My phone is +44 7700 900123, see https://example.org/doc',
    'SSN 123-45-6789 is sensitive, obviously.',
    'No PII here -- just a reflection on cognitive load.',
]
for s in samples:
    cleaned, counts = scrub(s)
    print(counts, '->', cleaned)

## 6.3 Verifying the sanitiser on an adversarial test-suite

In [ ]:
ADVERSARIAL = [
    'tamer.a.tes@example.co.uk',              # must match EMAIL
    '4532-1488-0343-6467',                    # test card
    '(415) 555-0199',                         # US phone
    'http://totally.safe.site/path?q=1',      # URL
    '987-65-4320',                            # SSN-like
]
passed = 0
for s in ADVERSARIAL:
    _, counts = scrub(s)
    assert counts, f'LEAK on {s!r}'
    passed += 1
print(f'{passed}/{len(ADVERSARIAL)} adversarial inputs scrubbed')

## 6.4 Fernet round-trip

Fernet composes AES-128-CBC with HMAC-SHA256 under a 256-bit key, and prepends a random 128-bit IV. It is authenticated, which matters: a bit-flipped ciphertext is rejected loudly rather than silently decrypted to garbage.

In [ ]:
try:
    from cryptography.fernet import Fernet, InvalidToken
    CRYPTO_OK = True
except Exception as e:
    CRYPTO_OK = False
    print(f'cryptography not available: {e}')

if CRYPTO_OK:
    key = Fernet.generate_key()
    box = Fernet(key)
    plain = json.dumps({'summary': 'reflected on pacing', 'emotion': 'calm'}).encode()
    ct = box.encrypt(plain)
    pt = box.decrypt(ct)
    assert pt == plain
    print(f'plain  : {plain}')
    print(f'cipher : {ct[:48]}...  (len={len(ct)})')
    print(f'round-trip ok')

    # Tampering test
    bad = bytearray(ct); bad[-1] ^= 0x01
    try:
        box.decrypt(bytes(bad))
        print('UNEXPECTED: tampered ciphertext decrypted')
    except InvalidToken:
        print('tampered ciphertext correctly rejected')

## 6.5 End-to-end: write a diary row, read it back

In [ ]:
if CRYPTO_OK:
    import time as _time
    summary_plain = 'user paused 2.3s, expressed mild frustration'
    cleaned, _ = scrub(summary_plain)
    enc = box.encrypt(cleaned.encode())
    conn.execute(
        'INSERT INTO diary (ts, user_id, summary, emotion, topics) VALUES (?,?,?,?,?)',
        (_time.time(), 'tamer', enc, 'frustration', json.dumps(['pacing']))
    )
    row = conn.execute(
        'SELECT summary, emotion, topics FROM diary LIMIT 1'
    ).fetchone()
    print('stored summary cipher :', row[0][:48], '...')
    print('stored emotion        :', row[1])
    print('stored topics         :', row[2])
    print('decrypted summary     :', box.decrypt(row[0]).decode())

## 6.6 Privacy invariants we rely on

1. The schema **never** declares a raw-text column (checked via PRAGMA).
2. All user-originating strings pass through `scrub` before summarisation.
3. Summary payloads are encrypted with Fernet before hitting disk; tampering is detected.
4. The bandit and user-model components consume only numeric feature vectors, so even an exfiltration of their internal state does not reveal content.